Decoder Only Plus Ollama

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Read the CSV file
df = pd.read_csv('../large_idiom_set_fr_eng.csv')


df['french_text'] = df['original_text']
df['english_text'] = df['text']

In [2]:
import ollama
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load your dataframe (assuming 'df' exists from your previous code)
# df = pd.read_csv("your_data.csv") 

def get_ollama_embeddings(texts, model="qwen2.5:0.5b"):
    embeddings = []
    # Ollama processes one by one (or you can parallelize with threads), 
    # but for 0.5B it is extremely fast.
    for text in tqdm(texts, desc="Generating Embeddings"):
        # The API call handles the GPU offloading automatically
        response = ollama.embeddings(model=model, prompt=text)
        embeddings.append(response["embedding"])
    return np.array(embeddings)

# Generate
print("Generating English embeddings...")
english_embeddings = get_ollama_embeddings(df['english_text'].tolist())

print("Generating French embeddings...")
french_embeddings = get_ollama_embeddings(df['french_text'].tolist())

# Create separate DataFrames with embeddings
english_df = pd.DataFrame({
    'text': df['english_text'],
    'language': 'English',
    'embedding': list(english_embeddings)
})

french_df = pd.DataFrame({
    'text': df['french_text'],
    'language': 'French',
    'embedding': list(french_embeddings)
})

# Combine both DataFrames
combined_df = pd.concat([english_df, french_df], ignore_index=True)



Generating English embeddings...


Generating Embeddings: 100%|██████████| 6616/6616 [01:09<00:00, 95.88it/s] 


Generating French embeddings...


Generating Embeddings: 100%|██████████| 6616/6616 [01:09<00:00, 95.38it/s] 


In [3]:
# Reduce embeddings to 2D using UMAP
from umap import UMAP
import numpy as np
# Reduce embeddings to 2D using UMAP
print("Reducing embeddings to 2D...")
all_embeddings = np.vstack(combined_df['embedding'].values)
reducer = UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(all_embeddings)

# Add 2D coordinates to dataframe
combined_df['x'] = embeddings_2d[:, 0]
combined_df['y'] = embeddings_2d[:, 1]

# Save to CSV for online visualization
output_df = combined_df[['text', 'language', 'x', 'y']].copy()
output_df.to_csv('embeddings_2d.csv', index=False)
print("Saved embeddings to 'embeddings_2d.csv'")

# Also save with full embeddings in parquet format
combined_df.to_parquet('embeddings_full.parquet')
print("Saved full embeddings to 'embeddings_full.parquet'")

print(f"\nDataset info: {len(output_df)} points total")
print(f"English: {len(output_df[output_df['language']=='English'])} points")
print(f"French: {len(output_df[output_df['language']=='French'])} points")



Reducing embeddings to 2D...


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved embeddings to 'embeddings_2d.csv'
Saved full embeddings to 'embeddings_full.parquet'

Dataset info: 13232 points total
English: 6616 points
French: 6616 points


In [4]:
# Calculate distances between English and French points
english_points = embeddings_2d[:len(english_embeddings)]
french_points = embeddings_2d[len(english_embeddings):]

distances = []
for i in range(len(english_points)):
    dist = np.sqrt((english_points[i][0] - french_points[i][0])**2 + 
                   (english_points[i][1] - french_points[i][1])**2)
    distances.append(dist)

avg_distance = np.mean(distances)

print(f"\nAverage distance between English and French: {avg_distance:.4f}")
print(f"Min distance: {np.min(distances):.4f}")
print(f"Max distance: {np.max(distances):.4f}")


Average distance between English and French: 20.8492
Min distance: 0.6909
Max distance: 26.9123


In Terminal : embedding-atlas Code2Qwen/embeddings_full.parquet --x x --y y --text text